# Module 1 — Stereo Camera Calibration & 3D Reconstruction

## What this notebook covers

This notebook reimplements **Part 1 of the assignment** (Exercises 1–3) in Python using OpenCV.
Instead of clicking through MATLAB's Stereo Camera Calibrator GUI, every step is written
explicitly so you can see exactly what the GUI was doing behind the scenes.

**Exercises covered:**
- Exercise 1: Stereo calibration → intrinsic K matrices, distortion coefficients
- Exercise 2: Single-image pose estimation → camera-to-probe distance
- Exercise 3: Disparity + Q matrix projection → 3D target coordinates

---

## Prerequisites

Download the **SERV-CT dataset** and place it at `data/raw/serv_ct/`.
See `data/README.md` for the download link.

The SERV-CT dataset was chosen because:
- It is real ex vivo porcine tissue filmed with a stereo endoscope — directly
  analogous to the surgical robot context of the assignment
- It includes calibration images, stereo image pairs, AND ground-truth depth
  from CT — so we can validate our results
- It is open-source and citable

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from parent directory

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from module1_stereo.calibration import (
    find_checkerboard_corners,
    calibrate_stereo,
    print_calibration_summary,
)
from module1_stereo.pose_estimation import estimate_checkerboard_pose, draw_pose_axes
from module1_stereo.disparity import compute_disparity_sgbm, point_disparity
from module1_stereo.reconstruction import (
    reproject_single_point,
    reproject_to_3d,
    filter_point_cloud,
)

print(f'OpenCV version: {cv2.__version__}')

## Step 1 — Stereo Calibration

### Why calibration is necessary

A camera is not a perfect pinhole. Two problems exist:

**Lens distortion:** The glass bends light rays. Straight lines in the world
appear curved in the image. Before we can do any geometry (measuring distances,
computing depths), we need to know — and correct for — this distortion.

**Stereo geometry:** Two cameras are never perfectly aligned. Even 0.1° of
rotation between cameras means corresponding points no longer sit on the same
horizontal scanline, and the disparity computation breaks down. Calibration
measures the exact rotation (R) and translation (T) between the cameras.

### What a checkerboard gives us

We use a checkerboard because:
1. Its geometry is known exactly (we measure the square size with a ruler)
2. Corners are high-contrast and easy to detect to sub-pixel accuracy
3. Showing it at many poses (angles and distances) over-constrains the problem,
   making the solution robust

The calibration algorithm asks: *what camera parameters would produce exactly
these observed corner positions, given where those corners are in 3D space?*
It solves this as a least-squares optimisation.

In [ ]:
# ── Dataset paths ──────────────────────────────────────────────────────────
# Adjust these to match where you placed the SERV-CT data.
DATA_ROOT = Path('../data/raw/serv_ct')

left_calib_images  = sorted((DATA_ROOT / 'calibration' / 'left').glob('*.png'))
right_calib_images = sorted((DATA_ROOT / 'calibration' / 'right').glob('*.png'))

print(f'Left calibration images:  {len(left_calib_images)}')
print(f'Right calibration images: {len(right_calib_images)}')

In [ ]:
# ── Visualise one calibration image pair ───────────────────────────────────
# Before running calibration it's always worth looking at the raw input.
# Check: is the checkerboard clearly visible? Is the image sharp?
# Blurry calibration images are one of the most common sources of error.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, path, label in zip(axes, [left_calib_images[0], right_calib_images[0]],
                            ['Left camera', 'Right camera']):
    img = cv2.imread(str(path))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(label)
    ax.axis('off')
plt.suptitle('Sample calibration image pair', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Run stereo calibration ─────────────────────────────────────────────────
# SERV-CT uses a standard calibration checkerboard.
# Check the dataset documentation for the exact pattern size and square size.

PATTERN_SIZE   = (9, 6)    # (interior cols, interior rows) — adjust per dataset
SQUARE_SIZE_MM = 3.0       # physical square size in mm — adjust per dataset

result = calibrate_stereo(
    left_calib_images,
    right_calib_images,
    PATTERN_SIZE,
    SQUARE_SIZE_MM,
)

print_calibration_summary(result)

### Reading the calibration output

**Reprojection error** (shown above) is the key quality indicator.
This is the average pixel distance between where the calibration algorithm
*predicted* corners would be (using the estimated parameters) and where
they were *actually detected*. Below 0.5 px is good; below 0.3 px is excellent.

**K matrix** — the intrinsic matrix:
```
K = [[fx,  0, cx],
     [ 0, fy, cy],
     [ 0,  0,  1]]
```
- `fx`, `fy`: focal lengths in pixels. For a square sensor, `fx ≈ fy`.
  Typical endoscope values are ~700–1500 px depending on sensor size.
- `cx`, `cy`: the principal point — where the optical axis meets the sensor.
  Should be close to the image centre (e.g., 640, 480 for a 1280×960 image).

**Radial distortion (k1, k2):** Endoscopic lenses are wide-angle and often
have significant barrel distortion (k1 < 0). A k1 of -0.3 means the image
centre is pulled inward by ~30% at the image edges — substantial but typical.

In [ ]:
# ── Visualise detected corners on calibration images ───────────────────────
# This is a sanity check: if corners are drawn correctly on the checkerboard,
# corner detection worked. Misaligned or missing corners indicate a problem
# with the pattern_size parameter or image quality.

img = cv2.imread(str(left_calib_images[0]))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
cols, rows = PATTERN_SIZE
found, corners = cv2.findChessboardCorners(gray, (cols, rows))

if found:
    cv2.drawChessboardCorners(img, (cols, rows), corners, found)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title('Detected checkerboard corners (left camera)')
    plt.axis('off')
    plt.show()
else:
    print('Checkerboard not found — check PATTERN_SIZE matches your images')

## Step 2 — Pose Estimation (Camera-to-Probe Distance)

### What Exercise 2 asked

Given a *single* rectified image of a 1mm checkerboard at the probe tip,
estimate the distance from the camera to the probe.

### Why this is possible from one image

With a single point, depth is ambiguous (a small close object looks identical
to a large far object). But a *planar target with known geometry* gives us many
points with known 3D relationships — enough to solve for the full 6-DOF pose.

The algorithm (solvePnP) finds the rotation and translation of the board
relative to the camera. The translation vector's magnitude = distance in mm.

In [ ]:
# ── Single-image pose estimation ───────────────────────────────────────────
# We use the left stereo camera's rectified image and its calibrated K matrix.
# Because the image is already rectified (undistorted), we pass zero distortion.

probe_image_path = DATA_ROOT / 'probe_image' / 'left_rectified.png'
PROBE_SQUARE_MM  = 1.0   # the probe checkerboard has 1mm squares
PROBE_PATTERN    = (4, 4) # adjust to the actual probe checkerboard size

pose = estimate_checkerboard_pose(
    image_path    = probe_image_path,
    K             = result.left.K,
    dist_coeffs   = np.zeros((5, 1)),  # zero because image is pre-rectified
    pattern_size  = PROBE_PATTERN,
    square_size_mm= PROBE_SQUARE_MM,
)

if pose:
    print(f'Distance to probe:       {pose.distance_mm:.2f} mm')
    print(f'Translation vector:      {pose.tvec.ravel()}')
    print(f'  tx={pose.tvec[0,0]:.2f} mm  (left-right)')
    print(f'  ty={pose.tvec[1,0]:.2f} mm  (up-down)')
    print(f'  tz={pose.tvec[2,0]:.2f} mm  (depth along optical axis)')
    print(f'Reprojection error:      {pose.reprojection_error_px:.4f} px')
else:
    print('Pose estimation failed — check PROBE_PATTERN and image path')

In [ ]:
# ── Visualise pose axes on the probe image ─────────────────────────────────
# Drawing XYZ axes at the checkerboard origin confirms the pose is correct.
# Red=X, Green=Y, Blue=Z (Z pointing toward camera is correct for a board
# facing the camera).

if pose:
    img = cv2.imread(str(probe_image_path))
    img_axes = draw_pose_axes(
        img, result.left.K, np.zeros((5,1)), pose, axis_length_mm=5.0
    )
    plt.figure(figsize=(10, 7))
    plt.imshow(cv2.cvtColor(img_axes, cv2.COLOR_BGR2RGB))
    plt.title(f'Probe pose — distance = {pose.distance_mm:.1f} mm')
    plt.axis('off')
    plt.show()

## Step 3 — Disparity Map & 3D Reconstruction

### Manual single-point reconstruction (Exercise 3 equivalent)

In Exercise 3 you manually identified the pixel coordinates of a tissue
target in both the left and right rectified images, then computed:

```
d = x_left - x_right
3D = Q @ [x, y, d, 1]^T   then divide by W
```

We do the same thing here programmatically, then extend it to a *dense* map
that computes depth for every pixel simultaneously.

In [ ]:
# ── Manual single-point reconstruction ────────────────────────────────────
# Identify the target pixel in each rectified image.
# You can find these by opening the images in any viewer and reading off
# the pixel coordinates (column, row) of the target region.

# Example values — replace with your actual pixel picks from the images:
target_left  = (542, 310)  # (col, row) in left rectified image
target_right = (518, 310)  # (col, row) in right rectified image
# Note: row (y) should be the SAME in both images after rectification.
# If y values differ significantly, the images may not be properly rectified.

d = point_disparity(target_left, target_right)
print(f'Disparity: d = {target_left[0]} - {target_right[0]} = {d:.1f} px')

coords_3d = reproject_single_point(
    x=target_left[0],
    y=target_left[1],
    disparity=d,
    Q=result.Q,
)
print(f'3D position of target: X={coords_3d[0]:.2f} mm, '
      f'Y={coords_3d[1]:.2f} mm, Z={coords_3d[2]:.2f} mm')
print(f'Distance from camera:  {np.linalg.norm(coords_3d):.2f} mm')

### Interpreting the 3D coordinates

The output (X, Y, Z) is in the coordinate frame of the **left camera**:
- X: horizontal position (positive = right of camera axis)
- Y: vertical position (positive = below camera axis)
- Z: depth — how far the target is from the camera along the optical axis

For a surgical robot, Z is the most important value — it tells the robot
how far to extend the instrument to reach the target tissue.

In [ ]:
# ── Dense disparity map ────────────────────────────────────────────────────
# Load a stereo image pair and compute depth at every pixel.

left_img  = cv2.imread(str(DATA_ROOT / 'stereo_pairs' / 'left_rect_000.png'))
right_img = cv2.imread(str(DATA_ROOT / 'stereo_pairs' / 'right_rect_000.png'))

disp_result = compute_disparity_sgbm(left_img, right_img)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(left_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Left rectified image')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(right_img, cv2.COLOR_BGR2RGB))
axes[1].set_title('Right rectified image')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(disp_result.disparity_visual, cv2.COLOR_BGR2RGB))
axes[2].set_title('Disparity map (warm=close, cool=far)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

valid_px = np.sum(~np.isnan(disp_result.disparity_map))
total_px = disp_result.disparity_map.size
print(f'Valid disparity pixels: {valid_px}/{total_px} ({100*valid_px/total_px:.1f}%)')

In [ ]:
# ── 3D Point cloud construction ────────────────────────────────────────────
# Apply the Q matrix to every pixel to get a full 3D scene.
# This extends the single-point Exercise 3 calculation to the entire image.

points_3d = reproject_to_3d(disp_result.disparity_map, result.Q)
pts, colors = filter_point_cloud(points_3d, left_img, z_min_mm=10, z_max_mm=150)

print(f'Point cloud size: {len(pts):,} points')
print(f'Depth range:  Z_min={pts[:,2].min():.1f} mm, Z_max={pts[:,2].max():.1f} mm')

In [ ]:
# ── Visualise point cloud ──────────────────────────────────────────────────
# We subsample to 10k points for a fast matplotlib scatter plot.
# The full point cloud visualisation is in the Streamlit dashboard (Module 3).

rng = np.random.default_rng(42)
idx = rng.choice(len(pts), size=min(10_000, len(pts)), replace=False)
sample_pts = pts[idx]
sample_col = colors[idx]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(sample_pts[:,0], sample_pts[:,1], sample_pts[:,2],
           c=sample_col, s=0.5, alpha=0.6)

# Mark the manually-identified tissue target
ax.scatter(*coords_3d, color='lime', s=200, marker='*',
           label=f'Target ({coords_3d[2]:.1f} mm depth)', zorder=5)

ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
ax.set_zlabel('Z depth (mm)')
ax.set_title('3D Scene Reconstruction')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Step | Assignment | This notebook |
|------|-----------|---------------|
| Stereo calibration | MATLAB GUI (Stereo Camera Calibrator) | `cv2.stereoCalibrate()` |
| K matrix + distortion | Read from GUI output | `result.left.K`, `result.left.dist_coeffs` |
| Probe distance | `estimateCheckerboardPose` | `cv2.solvePnP()` |
| Disparity (single point) | Manual: `d = x_left - x_right` | `point_disparity()` |
| 3D reconstruction | Manual: `Q @ [x,y,d,1]^T` | `reproject_single_point()` |
| Dense depth map | Not in assignment | `compute_disparity_sgbm()` |
| Point cloud | Not in assignment | `reproject_to_3d()` + Open3D (Module 3) |

**Next:** `02_3d_reconstruction.ipynb` — working with the Hamlyn Centre in vivo dataset
and visualising the full 3D scene in Open3D.